# 08 CNN-GRU


Train a compact convolutional-recurrent model that extracts local temporal patterns before GRU sequence modeling.


In [ ]:
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd / "wesad_stress_project").exists():
    PROJECT_ROOT = cwd / "wesad_stress_project"
else:
    PROJECT_ROOT = cwd
PROJECT_ROOT


## 2. Training utility functions


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def logits_to_probabilities(logits: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(logits.reshape(-1))


def collect_probabilities(model: torch.nn.Module, loader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probabilities: List[np.ndarray] = []
    labels: List[np.ndarray] = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x)
            probabilities.append(logits_to_probabilities(logits).cpu().numpy())
            labels.append(batch_y.float().cpu().numpy().reshape(-1))
    return np.concatenate(probabilities), np.concatenate(labels)


def select_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> Tuple[float, pd.DataFrame]:
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.01):
        predicted = (probabilities >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(round(threshold, 2)),
                "macro_f1": float(f1_score(y_true, predicted, average="macro", zero_division=0)),
            }
        )
    table = pd.DataFrame(rows)
    best_row = table.sort_values(["macro_f1", "threshold"], ascending=[False, True]).iloc[0]
    return float(best_row["threshold"]), table


def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> Dict[str, object]:
    predicted = (probabilities >= threshold).astype(int)
    precision, recall, _, _ = precision_recall_fscore_support(
        y_true, predicted, labels=[0, 1], zero_division=0
    )
    metrics: Dict[str, object] = {
        "macro_f1": float(f1_score(y_true, predicted, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, predicted, average="weighted", zero_division=0)),
        "non_stress_precision": float(precision[0]),
        "non_stress_recall": float(recall[0]),
        "stress_precision": float(precision[1]),
        "stress_recall": float(recall[1]),
        "confusion_matrix": confusion_matrix(y_true, predicted, labels=[0, 1]).tolist(),
    }
    metrics["roc_auc"] = _safe_score(roc_auc_score, y_true, probabilities)
    metrics["average_precision"] = _safe_score(average_precision_score, y_true, probabilities)
    return metrics


def _safe_score(metric_fn, y_true: np.ndarray, probabilities: np.ndarray) -> float | None:
    try:
        return float(metric_fn(y_true, probabilities))
    except ValueError:
        return None


def per_subject_metrics(
    metadata: pd.DataFrame, y_true: np.ndarray, probabilities: np.ndarray, threshold: float
) -> pd.DataFrame:
    rows = []
    for subject_id, group in metadata.reset_index().groupby("subject_id"):
        indices = group["index"].to_numpy()
        metrics = binary_metrics(y_true[indices], probabilities[indices], threshold)
        metrics.pop("confusion_matrix", None)
        rows.append({"subject_id": subject_id, **metrics, "n_windows": int(len(indices))})
    return pd.DataFrame(rows)


def train_one_epoch(
    model: torch.nn.Module,
    loader,
    criterion,
    optimizer,
    device: torch.device,
    gradient_clip: float | None = None,
) -> float:
    model.train()
    losses = []
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.float().to(device).reshape(-1)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x).reshape(-1)
        loss = criterion(logits, batch_y)
        loss.backward()
        if gradient_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def evaluate_loss(model: torch.nn.Module, loader, criterion, device: torch.device) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device).reshape(-1)
            logits = model(batch_x).reshape(-1)
            loss = criterion(logits, batch_y)
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def train_with_early_stopping(
    model: torch.nn.Module,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device: torch.device,
    max_epochs: int = 50,
    patience: int = 8,
    gradient_clip: float | None = None,
) -> Tuple[torch.nn.Module, pd.DataFrame, Dict[str, torch.Tensor]]:
    best_state = None
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, gradient_clip=gradient_clip)
        validation_loss = evaluate_loss(model, validation_loader, criterion, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
            }
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_state or {}


def pos_weight_from_labels(y_train: torch.Tensor, device: torch.device) -> torch.Tensor:
    y = y_train.reshape(-1)
    number_negative = (y == 0).sum().item()
    number_positive = (y == 1).sum().item()
    if number_positive == 0:
        raise ValueError("Cannot compute pos_weight because the training set has no positive labels.")
    return torch.tensor([number_negative / number_positive], dtype=torch.float32, device=device)


def count_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def save_json(path: Path, data: Dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)


def save_model_artifacts(
    artifact_dir: Path,
    model: torch.nn.Module,
    model_config: Dict[str, object],
    threshold: float,
    history: pd.DataFrame,
    validation_metrics: Dict[str, object],
    test_metrics: Dict[str, object],
    per_subject: pd.DataFrame,
    test_predictions: pd.DataFrame,
) -> None:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), artifact_dir / "best_model.pt")
    save_json(artifact_dir / "model_config.json", model_config)
    save_json(artifact_dir / "threshold.json", {"threshold": threshold})
    save_json(artifact_dir / "validation_metrics.json", validation_metrics)
    save_json(artifact_dir / "test_metrics.json", test_metrics)
    history.to_csv(artifact_dir / "training_history.csv", index=False)
    per_subject.to_csv(artifact_dir / "per_subject_metrics.csv", index=False)
    test_predictions.to_csv(artifact_dir / "test_predictions.csv", index=False)


def prediction_table(
    metadata: pd.DataFrame,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "window_id": metadata["window_id"].to_numpy(),
            "subject_id": metadata["subject_id"].to_numpy(),
            "true_label": y_true.astype(int),
            "stress_probability": probabilities,
            "predicted_label": (probabilities >= threshold).astype(int),
            "threshold": threshold,
        }
    )


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
MAX_EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
device


## 3. CNN-GRU model definition


In [ ]:
from __future__ import annotations

import torch
from torch import nn


class CNNGRUClassifier(nn.Module):
    def __init__(self, input_channels: int = 6, conv_channels: int = 32, hidden_size: int = 64, dropout: float = 0.3) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, conv_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        self.gru = nn.GRU(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        _, hidden = self.gru(x)
        return self.classifier(hidden[-1])


## 4. Load sequence tensors


In [ ]:
sequence_dir = PROJECT_ROOT / "data" / "processed" / "sequence"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"

X_train = torch.load(sequence_dir / "X_train.pt").float()
y_train = torch.load(sequence_dir / "y_train.pt").float()
X_validation = torch.load(sequence_dir / "X_validation.pt").float()
y_validation = torch.load(sequence_dir / "y_validation.pt").float()
X_test = torch.load(sequence_dir / "X_test.pt").float()
y_test = torch.load(sequence_dir / "y_test.pt").float()

metadata_validation = pd.read_csv(metadata_dir / "windows_validation.csv")
metadata_test = pd.read_csv(metadata_dir / "windows_test.csv")

assert len(metadata_validation) == len(y_validation)
assert len(metadata_test) == len(y_test)
assert np.array_equal(metadata_validation["binary_label"].to_numpy(), y_validation.numpy().astype(int))
assert np.array_equal(metadata_test["binary_label"].to_numpy(), y_test.numpy().astype(int))
assert metadata_test["subject_id"].isin(["S2", "S11", "S14"]).all()

X_train.shape, X_validation.shape, X_test.shape


## 5. Load window metadata


In [ ]:
metadata_validation.head(), metadata_test.head()


## 6. Create TensorDataset objects


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)


## 7. Create DataLoaders


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 8. Inspect input and output shapes


In [ ]:
model = CNNGRUClassifier(input_channels=X_train.shape[2]).to(device)
print(X_train[:4].shape)
print(model(X_train[:4].to(device)).shape)


## 9. Define loss and optimizer


In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


## 10. Train without imbalance correction


In [ ]:
set_seed(42)
unweighted_model = CNNGRUClassifier(input_channels=X_train.shape[2]).to(device)
unweighted_criterion = torch.nn.BCEWithLogitsLoss()
unweighted_optimizer = torch.optim.Adam(unweighted_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
unweighted_model, unweighted_history, _ = train_with_early_stopping(unweighted_model, train_loader, validation_loader, unweighted_criterion, unweighted_optimizer, device, MAX_EPOCHS, PATIENCE, gradient_clip=1.0)
unweighted_history.tail()


## 11. Train with class weighting


In [ ]:
set_seed(42)
model = CNNGRUClassifier(input_channels=X_train.shape[2]).to(device)
pos_weight = pos_weight_from_labels(y_train, device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model, history, best_state = train_with_early_stopping(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    gradient_clip=1.0,
)
history.tail()


## 12. Use early stopping


In [ ]:
unweighted_validation_probabilities, unweighted_validation_true = collect_probabilities(unweighted_model, validation_loader, device)
unweighted_threshold, _ = select_threshold(unweighted_validation_true, unweighted_validation_probabilities)
unweighted_validation_metrics = binary_metrics(
    unweighted_validation_true,
    unweighted_validation_probabilities,
    unweighted_threshold,
)

weighted_validation_probabilities, weighted_validation_true = collect_probabilities(model, validation_loader, device)
weighted_threshold, _ = select_threshold(weighted_validation_true, weighted_validation_probabilities)
weighted_validation_metrics = binary_metrics(
    weighted_validation_true,
    weighted_validation_probabilities,
    weighted_threshold,
)

unweighted_best_epoch = int(unweighted_history.loc[unweighted_history["validation_loss"].idxmin(), "epoch"])
weighted_best_epoch = int(history.loc[history["validation_loss"].idxmin(), "epoch"])

validation_variant_comparison = pd.DataFrame(
    [
        {
            "method": "no_correction",
            "best_epoch": unweighted_best_epoch,
            "threshold": unweighted_threshold,
            "macro_f1": unweighted_validation_metrics["macro_f1"],
            "stress_recall": unweighted_validation_metrics["stress_recall"],
            "average_precision": unweighted_validation_metrics["average_precision"],
        },
        {
            "method": "class_weight",
            "best_epoch": weighted_best_epoch,
            "threshold": weighted_threshold,
            "macro_f1": weighted_validation_metrics["macro_f1"],
            "stress_recall": weighted_validation_metrics["stress_recall"],
            "average_precision": weighted_validation_metrics["average_precision"],
        },
    ]
)
display(validation_variant_comparison)

if weighted_validation_metrics["macro_f1"] >= unweighted_validation_metrics["macro_f1"]:
    selected_model = model
    selected_history = history
    selected_threshold = weighted_threshold
    selected_validation_probabilities = weighted_validation_probabilities
    selected_validation_true = weighted_validation_true
    selected_validation_metrics = weighted_validation_metrics
    selected_imbalance_method = "class_weight"
    selected_best_epoch = weighted_best_epoch
else:
    selected_model = unweighted_model
    selected_history = unweighted_history
    selected_threshold = unweighted_threshold
    selected_validation_probabilities = unweighted_validation_probabilities
    selected_validation_true = unweighted_validation_true
    selected_validation_metrics = unweighted_validation_metrics
    selected_imbalance_method = "no_correction"
    selected_best_epoch = unweighted_best_epoch

model = selected_model
history = selected_history
threshold = selected_threshold
validation_probabilities = selected_validation_probabilities
validation_true = selected_validation_true
validation_metrics = {
    **selected_validation_metrics,
    "selected_imbalance_method": selected_imbalance_method,
    "best_validation_epoch": selected_best_epoch,
    "variant_comparison": validation_variant_comparison.to_dict(orient="records"),
}

threshold, selected_imbalance_method, validation_metrics


## 13. Select validation threshold


In [ ]:
# The previous cell selected the imbalance method and threshold using validation macro-F1.
threshold, selected_imbalance_method, validation_metrics


## 14. Evaluate test data


In [ ]:
test_probabilities, test_true = collect_probabilities(model, test_loader, device)
test_metrics = binary_metrics(test_true, test_probabilities, threshold)
test_metrics


## 15. Evaluate each test subject


In [ ]:
subject_metrics = per_subject_metrics(metadata_test, test_true, test_probabilities, threshold)
subject_metrics


## 16. Plot training curves and confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
history.plot(x="epoch", y=["train_loss", "validation_loss"], ax=ax)
ax.set_title("Training history")
ax.set_ylabel("BCE loss")
plt.show()

cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non-stress", "stress"])
ax.set_yticks([0, 1], labels=["non-stress", "stress"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.show()


## 17. Plot ROC and precision-recall curves


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(test_true, test_probabilities, ax=axes[0])
PrecisionRecallDisplay.from_predictions(test_true, test_probabilities, ax=axes[1])
plt.tight_layout()
plt.show()


## 18. Save model and results


In [ ]:
artifact_dir = PROJECT_ROOT / "artifacts" / "models" / "cnn_gru"
test_predictions = prediction_table(metadata_test, test_true, test_probabilities, threshold)

save_model_artifacts(
    artifact_dir=artifact_dir,
    model=model,
    model_config={ "model": "CNNGRUClassifier", "input_channels": int(X_train.shape[2]), "conv_channels": 32, "hidden_size": 64, "dropout": 0.3,
        "parameter_count": count_parameters(model),
        "selected_imbalance_method": selected_imbalance_method,
        "best_validation_epoch": selected_best_epoch,
        "random_seed": 42, "uses_class_weighting": selected_imbalance_method == "class_weight" },
    threshold=threshold,
    history=history,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    per_subject=subject_metrics,
    test_predictions=test_predictions,
)
artifact_dir
